In [1]:
import pandas as pd

attack = pd.read_html("Dataset/attacking.html")[0]
defend = pd.read_html("Dataset/defend.html")[0]
pyshic = pd.read_html("Dataset/fisik.html")[0]
goalkeeping = pd.read_html("Dataset/gk.html")[0]
mental = pd.read_html("Dataset/mental.html")[0]
technical = pd.read_html("Dataset/technical.html")[0]
general = pd.read_html("Dataset/general.html")[0]
position = pd.read_html("Dataset/position.html")[0]
contract = pd.read_html("Dataset/contract.html")[0]
#MERGING DATA

print(attack.columns.tolist())
print(goalkeeping.columns.tolist())

source_tables = [attack, defend, pyshic, goalkeeping, mental, technical, general, position, contract]
for table in source_tables:
    table["_name_occurrence"] = table.groupby("Name", dropna=False).cumcount()

merge_keys = ["Name", "_name_occurrence"]
common_columns = ["Rec", "Inf", "WR", "Transfer Value"]
df = general.copy()

for table in [attack, defend, pyshic, goalkeeping, mental, technical, position, contract]:
    table = table.drop(
        columns=[column for column in common_columns if column in table.columns]
    )
    df = df.merge(
        table,
        on=merge_keys,
        how="left",
        validate="one_to_one",
        suffixes=("", "_source")
    )

source_columns = [column for column in df.columns if column.endswith("_source")]
for source_column in source_columns:
    original_column = source_column.removesuffix("_source")
    if original_column in df.columns:
        df[original_column] = df[original_column].combine_first(df[source_column])
        df.drop(columns=source_column, inplace=True)

df = df.drop(columns="_name_occurrence")
df.insert(0, "Player ID", range(1, len(df) + 1))

print(f"Merged rows: {len(df)}")
print(f"Unique player names: {df['Name'].nunique()}")
print(f"Duplicate names from distinct players: {df['Name'].duplicated().sum()}")
print(f"Remaining source columns: {[column for column in df.columns if column.endswith('_source')]}")



df.to_csv("Dataset/fm24_players.csv", index=False)

print(df.head())

['Rec', 'Inf', 'Name', 'Cro', 'Dri', 'Fre', 'Fin', 'Fir', 'Fla', 'Lon', 'OtB', 'Pas', 'Vis', 'Transfer Value']
['Rec', 'Inf', 'Name', 'Aer', 'Cmd', 'Com', 'Ecc', 'Han', 'Kic', '1v1', 'Ref', 'TRO', 'Pun', 'Thr', 'Transfer Value']
Merged rows: 1181
Unique player names: 1175
Duplicate names from distinct players: 6
Remaining source columns: []


PermissionError: [Errno 13] Permission denied: 'Dataset/fm24_players.csv'

In [2]:
import pandas as pd
import re


def convert(value):
    if pd.isna(value):
        return 0.0

    matches = re.findall(r"(\d+(?:[.,]\d+)?)\s*([KM])", str(value).upper())
    if not matches:
        return 0.0

    converted_values = []
    for number, unit in matches:
        amount = float(number.replace(",", "."))
        multiplier = 1_000_000 if unit == "M" else 1_000
        converted_values.append(amount * multiplier)

    return max(converted_values)


def wageConvert(value):
    if pd.isna(value):
        return 0.0

    match = re.search(r"[\d.,]+", str(value))
    if not match:
        return 0.0

    number = match.group().replace(",", "")
    return float(number)


def changeType(col):
    return col.astype(float)


def drop(df):
    include = ["Rec", "Inf", "WR", "Ability", "Potential"]
    return df.drop(columns=[column for column in include if column in df.columns])


exclude = [
    "Name", "Club", "Nat", "Height", "Weight", "Best Pos", "Best Role",
    "Expires", "Agreed Playing Time", "F/PT",
]

df = pd.read_csv("Dataset/fm24_players.csv")
df["Transfer Value"] = df["Transfer Value"].apply(convert)
df["Wage"] = df["Wage"].apply(wageConvert)
df = drop(df)

print(df.info())
print(df.iloc[0].to_string())

df.to_csv("Dataset/CleanFm24_players.csv", index=False, encoding="utf-8-sig")

<class 'pandas.DataFrame'>
RangeIndex: 1181 entries, 0 to 1180
Data columns (total 62 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Player ID            1181 non-null   int64  
 1   Name                 1181 non-null   str    
 2   Position             1181 non-null   str    
 3   Club                 1181 non-null   str    
 4   Nat                  1181 non-null   str    
 5   Height               1181 non-null   str    
 6   Weight               1181 non-null   str    
 7   Age                  1181 non-null   int64  
 8   Transfer Value       1181 non-null   float64
 9   Cro                  1181 non-null   int64  
 10  Dri                  1181 non-null   int64  
 11  Fre                  1181 non-null   int64  
 12  Fin                  1181 non-null   int64  
 13  Fir                  1181 non-null   int64  
 14  Fla                  1181 non-null   int64  
 15  Lon                  1181 non-null   int64  
 16 

In [3]:
import pandas as pd

df = pd.read_csv(
    "Dataset/CleanFm24_players.csv",
    encoding="utf-8-sig"
)

target = "Best Role"

exclude = {
    "Player ID",
    "Name",
    "Club",
    "Nat",
    "Position",
    "Best Pos",
    "Best Role",
    "Best Duty",
    "Height",
    "Weight",
    "Expires",
    "Agreed Playing Time",
    "F/PT",
    "Transfer Value"
}

feature_columns = []

for column in df.columns:
    if column in exclude:
        continue

    numeric_column = pd.to_numeric(df[column], errors="coerce")

    if numeric_column.notna().sum() > 0:
        df[column] = numeric_column
        feature_columns.append(column)

X = df[feature_columns].fillna(0)
y = df[target]

valid_rows = y.notna() & y.ne("Unknown")
X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("Jumlah fitur:", len(feature_columns))
print("Daftar fitur:")
print(feature_columns)

print("\nUkuran X:", X.shape)
print("Ukuran y:", y.shape)
print("\nTipe data fitur:")
print(X.dtypes)

Jumlah fitur: 48
Daftar fitur:
['Age', 'Cro', 'Dri', 'Fre', 'Fin', 'Fir', 'Fla', 'Lon', 'OtB', 'Pas', 'Vis', 'Acc', 'Ant', 'Hea', 'Jum', 'Mar', 'Pac', 'Pos', 'Sta', 'Str', 'Tck', 'Agi', 'Bal', 'Aer', 'Cmd', 'Com', 'Ecc', 'Han', 'Kic', '1v1', 'Ref', 'TRO', 'Pun', 'Thr', 'Agg', 'Bra', 'Cmp', 'Cnt', 'Dec', 'Det', 'Ldr', 'Tea', 'Wor', 'Cor', 'L Th', 'Pen', 'Tec', 'Wage']

Ukuran X: (1181, 48)
Ukuran y: (1181,)

Tipe data fitur:
Age       int64
Cro       int64
Dri       int64
Fre       int64
Fin       int64
Fir       int64
Fla       int64
Lon       int64
OtB       int64
Pas       int64
Vis       int64
Acc       int64
Ant       int64
Hea       int64
Jum       int64
Mar       int64
Pac       int64
Pos       int64
Sta       int64
Str       int64
Tck       int64
Agi       int64
Bal       int64
Aer       int64
Cmd       int64
Com       int64
Ecc       int64
Han       int64
Kic       int64
1v1       int64
Ref       int64
TRO       int64
Pun       int64
Thr       int64
Agg       int64
Bra       in

In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder =LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

role_counts = y.value_counts()
valid_roles = role_counts[role_counts >= 2].index
valid_rows = y.isin(valid_roles)

X_model = X.loc[valid_rows].reset_index(drop=True)
y_model = y.loc[valid_rows].reset_index(drop=True)

label_encoder = LabelEncoder()
y_encoded_model = label_encoder.fit_transform(y_model)

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_encoded_model,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded_model
)

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    reg_lambda =2,
    subsample=0.8,
    objective="multi:softprob",
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train)
print("Model berhasil dilatih")
print("Jumlah kelas:", len(label_encoder.classes_))
print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)

Model berhasil dilatih
Jumlah kelas: 32
Data training: (942, 48)
Data testing: (236, 48)


In [15]:
from sklearn.metrics import accuracy_score

prediction = model.predict(X_test)
accuracy = accuracy_score(y_test, prediction)

print("Accuracy:", accuracy)

Accuracy: 0.6101694915254238


In [7]:
import numpy as np
import pandas as pd

ROLE_BY_POSITION = {
    "GK": ["Goalkeeper", "Sweeper Keeper"],
    "DR/DL": ["Full Back", "Wing Back", "Complete Wing Back", "Inverted Wing Back", "No-Nonsense Full Back", "Defensive Full Back", "Wide Centre-Back"],
    "DC": ["Central Defender", "Ball Playing Defender", "No-Nonsense Centre-Back", "Wide Centre-Back", "Libero"],
    "DM": ["Defensive Midfielder", "Anchor", "Half Back", "Deep Lying Playmaker", "Segundo Volante", "Regista", "Roaming Playmaker"],
    "MR/ML": ["Wide Midfielder", "Winger", "Defensive Winger", "Inverted Winger", "Wide Playmaker", "Wide Target Forward"],
    "MC": ["Central Midfielder", "Ball Winning Midfielder", "Box-to-Box Midfielder", "Deep Lying Playmaker", "Advanced Playmaker", "Mezzala", "Carrilero", "Roaming Playmaker", "Segundo Volante"],
    "AMR/AML": ["Winger", "Inverted Winger", "Inside Forward", "Wide Playmaker", "Advanced Playmaker", "Raumdeuter"],
    "AMC": ["Attacking Midfielder", "Advanced Playmaker", "Enganche", "Trequartista", "Shadow Striker", "Raumdeuter"],
    "STC": ["Advanced Forward", "Complete Forward", "Deep Lying Forward", "False Nine", "Pressing Forward", "Poacher", "Target Forward", "Trequartista"],
}


POSITION_ALIASES = {
    "GK": "GK", "DR": "DR/DL", "DL": "DR/DL", "DR/DL": "DR/DL",
    "DC": "DC", "DM": "DM", "MR": "MR/ML", "ML": "MR/ML", "MR/ML": "MR/ML",
    "MC": "MC", "AMR": "AMR/AML", "AML": "AMR/AML", "AMR/AML": "AMR/AML",
    "AMC": "AMC", "STC": "STC", "ST": "STC",
}


ROLE_PROFILES = {
    "Goalkeeper": {"Ref": 1.0, "Han": 0.8, "One": 0.8, "Com": 0.7, "Agi": 0.5},
    "Sweeper Keeper": {"Ref": 0.9, "Pas": 0.7, "Kic": 0.8, "Com": 0.8, "Agi": 0.6},
    "Full Back": {"Pac": 0.7, "Sta": 0.6, "Tck": 0.8, "Cro": 0.7, "Pos": 0.7},
    "Wing Back": {"Pac": 0.8, "Sta": 0.8, "Cro": 0.8, "Dri": 0.7, "Tck": 0.6},
    "Complete Wing Back": {"Pac": 0.8, "Sta": 0.9, "Cro": 0.8, "Dri": 0.8, "Tck": 0.7},
    "Inverted Wing Back": {"Pas": 0.8, "Tec": 0.7, "Dec": 0.8, "Pos": 0.8, "Dri": 0.6},
    "No-Nonsense Full Back": {"Tck": 0.9, "Mar": 0.8, "Str": 0.7, "Pos": 0.8},
    "Defensive Full Back": {"Tck": 0.9, "Mar": 0.9, "Pos": 0.9, "Pac": 0.5},
    "Wide Centre-Back": {"Pac": 0.7, "Tck": 0.8, "Pas": 0.6, "Pos": 0.8},
    "Central Defender": {"Tck": 0.9, "Mar": 0.8, "Hea": 0.7, "Pos": 0.9},
    "Ball Playing Defender": {"Tck": 0.8, "Pas": 0.8, "Tec": 0.7, "Dec": 0.8},
    "No-Nonsense Centre-Back": {"Tck": 0.9, "Hea": 0.8, "Str": 0.8, "Pos": 0.8},
    "Libero": {"Pas": 0.8, "Tec": 0.7, "Dec": 0.8, "Vis": 0.7, "Dri": 0.6},
    "Defensive Midfielder": {"Tck": 0.8, "Pos": 0.8, "Sta": 0.7, "Dec": 0.7},
    "Anchor": {"Tck": 0.9, "Pos": 0.9, "Str": 0.7, "Dec": 0.8},
    "Half Back": {"Tck": 0.8, "Pos": 0.8, "Pas": 0.7, "Dec": 0.8},
    "Deep Lying Playmaker": {"Pas": 0.9, "Vis": 0.8, "Tec": 0.7, "Dec": 0.8},
    "Segundo Volante": {"Sta": 0.8, "Pas": 0.7, "Dri": 0.7, "Fin": 0.6, "Tck": 0.6},
    "Regista": {"Pas": 0.9, "Vis": 0.9, "Tec": 0.8, "Dec": 0.7},
    "Roaming Playmaker": {"Pas": 0.8, "Vis": 0.8, "Sta": 0.8, "Dri": 0.7},
    "Wide Midfielder": {"Sta": 0.7, "Pas": 0.7, "Cro": 0.7, "Tck": 0.5},
    "Winger": {"Pac": 0.8, "Dri": 0.8, "Cro": 0.8, "Tec": 0.7},
    "Defensive Winger": {"Pac": 0.7, "Sta": 0.8, "Tck": 0.6, "Cro": 0.6},
    "Inverted Winger": {"Dri": 0.8, "Pas": 0.7, "Fin": 0.7, "Tec": 0.8},
    "Wide Playmaker": {"Pas": 0.9, "Vis": 0.8, "Tec": 0.8, "Dri": 0.7},
    "Wide Target Forward": {"Hea": 0.8, "Str": 0.8, "Pac": 0.5, "Cro": 0.6},
    "Central Midfielder": {"Pas": 0.7, "Sta": 0.7, "Dec": 0.7, "Tec": 0.6},
    "Ball Winning Midfielder": {"Tck": 0.9, "Agg": 0.8, "Sta": 0.8, "Wor": 0.8},
    "Box-to-Box Midfielder": {"Sta": 0.9, "Wor": 0.8, "Pas": 0.7, "Dri": 0.7, "Fin": 0.6},
    "Advanced Playmaker": {"Pas": 0.9, "Vis": 0.9, "Tec": 0.8, "Dri": 0.7},
    "Mezzala": {"Pas": 0.8, "Dri": 0.8, "Sta": 0.7, "Fin": 0.6},
    "Carrilero": {"Sta": 0.8, "Wor": 0.8, "Pas": 0.7, "Tck": 0.6},
    "Inside Forward": {"Dri": 0.8, "Fin": 0.8, "Pac": 0.8, "Tec": 0.7},
    "Attacking Midfielder": {"Pas": 0.8, "Dri": 0.7, "Fin": 0.7, "Vis": 0.7},
    "Enganche": {"Pas": 0.9, "Vis": 0.9, "Tec": 0.8, "Fin": 0.6},
    "Trequartista": {"Pas": 0.9, "Vis": 0.9, "Dri": 0.8, "Tec": 0.8},
    "Shadow Striker": {"Fin": 0.8, "Dri": 0.8, "Pac": 0.8, "Off": 0.8},
    "Raumdeuter": {"Off": 0.9, "Fin": 0.8, "Pac": 0.7, "Ant": 0.7},
    "Advanced Forward": {"Fin": 0.9, "Pac": 0.8, "Off": 0.9, "Dri": 0.7},
    "Complete Forward": {"Fin": 0.8, "Hea": 0.7, "Dri": 0.8, "Pas": 0.7, "Str": 0.7},
    "Deep Lying Forward": {"Pas": 0.8, "Vis": 0.7, "Fin": 0.7, "Str": 0.6},
    "False Nine": {"Pas": 0.8, "Vis": 0.8, "Dri": 0.8, "Tec": 0.8},
    "Pressing Forward": {"Wor": 0.9, "Sta": 0.8, "Agg": 0.8, "Pac": 0.7, "Fin": 0.7},
    "Poacher": {"Fin": 0.95, "Off": 0.9, "Ant": 0.8, "Pac": 0.6},
    "Target Forward": {"Hea": 0.9, "Str": 0.9, "Fin": 0.7, "Ant": 0.7},

}

PLAYSTYLE_BONUSES = {
    "possession": ["Deep Lying Playmaker", "Advanced Playmaker", "Regista", "Ball Playing Defender", "False Nine"],
    "gegenpress": ["Pressing Forward", "Ball Winning Midfielder", "Box-to-Box Midfielder", "Wing Back"],
    "counter": ["Advanced Forward", "Winger", "Inside Forward", "Shadow Striker", "Complete Wing Back"],
    "direct": ["Target Forward", "Wide Target Forward", "Poacher", "No-Nonsense Centre-Back"],
    "balanced": [],
}


def _role_attribute_score(players, role):
    profile = ROLE_PROFILES.get(role, {})
    available = {attribute: weight for attribute, weight in profile.items() if attribute in players.columns}
    if not available:
        return pd.Series(0.0, index=players.index)
    values = players[list(available)].apply(pd.to_numeric, errors="coerce").fillna(0).clip(0, 100)
    weights = pd.Series(available)
    return values.mul(weights, axis=1).sum(axis=1) / weights.sum()


def recommend_players(budget, position, playstyle="balanced", top_n=10, min_confidence=0):
    """Rumus: skor akhir = 70% atribut role + 20% playstyle + 10% confidence model."""
    position_key = POSITION_ALIASES.get(str(position).strip().upper())
    if position_key is None:
        raise ValueError(f"Posisi harus salah satu dari: {list(POSITION_ALIASES)}")
    playstyle_key = str(playstyle).strip().lower()
    if playstyle_key not in PLAYSTYLE_BONUSES:
        raise ValueError(f"Playstyle harus salah satu dari: {list(PLAYSTYLE_BONUSES)}")

    candidates = df[pd.to_numeric(df["Transfer Value"], errors="coerce").fillna(0) <= float(budget)].copy()
    if candidates.empty:
        return pd.DataFrame()

    role_scores = pd.DataFrame(index=candidates.index)
    for role in ROLE_BY_POSITION[position_key]:
        role_scores[role] = _role_attribute_score(candidates, role)
    best_role = role_scores.idxmax(axis=1)
    attribute_score = role_scores.max(axis=1)
    playstyle_score = best_role.isin(PLAYSTYLE_BONUSES[playstyle_key]).astype(float) * 100

    if hasattr(model, "predict_proba"):
        model_features = candidates.reindex(columns=feature_columns, fill_value=0).apply(pd.to_numeric, errors="coerce").fillna(0)
        probabilities = model.predict_proba(model_features)
        known_roles = list(label_encoder.classes_)
        model_confidence = pd.Series(probabilities.max(axis=1) * 100, index=candidates.index)
    else:
        model_confidence = pd.Series(0.0, index=candidates.index)
        known_roles = []

    candidates["Predicted Role"] = best_role
    candidates["Role Fit"] = attribute_score.round(2)
    candidates["Playstyle Fit"] = playstyle_score
    candidates["Model Confidence"] = model_confidence.round(2)
    candidates["Recommendation Score"] = (0.70 * candidates["Role Fit"] + 0.20 * candidates["Playstyle Fit"] + 0.10 * candidates["Model Confidence"]).round(2)

    result_columns = ["Name", "Club", "Transfer Value", "Position", "Predicted Role", "Role Fit", "Playstyle Fit", "Model Confidence", "Recommendation Score"]
    result_columns = [column for column in result_columns if column in candidates.columns]
    return candidates[candidates["Model Confidence"] >= min_confidence].sort_values("Recommendation Score", ascending=False)[result_columns].head(top_n)


# Input user: budget dalam mata uang dataset, posisi, gaya bermain, dan jumlah hasil.
budget = 80_000_000
position = "DM"
playstyle = "gegenpress"
top_n = 10

recommendations = recommend_players(budget, position, playstyle, top_n)
print(recommendations.to_string(index=False))

             Name                Club  Transfer Value         Position       Predicted Role  Role Fit  Playstyle Fit  Model Confidence  Recommendation Score
  Frenkie de Jong           Barcelona      71000000.0        DM, M (C)              Regista     17.06            0.0         94.059998                 21.35
 Antonio RÃ¼diger         Real Madrid      72000000.0            D (C)               Anchor     16.15            0.0         96.330002                 20.94
     Mario GÃ¶tze Eintracht Frankfurt       9600000.0         M/AM (C)              Regista     15.82            0.0         95.510002                 20.63
 Florentino LuÃ­s             Burnley      17000000.0        DM, M (C) Defensive Midfielder     16.07            0.0         91.809998                 20.43
      Dani Parejo          Villarreal        650000.0     DM, M/AM (C)              Regista     15.36            0.0         96.440002                 20.40
    FÃ¡bio Vieira             Hamburg      38000000.0  M (

In [8]:
def _position_mask(players, position_key):
    position_text = players["Position"].fillna("").astype(str).str.upper()
    patterns = {
        "GK": r"\bGK\b",
        "DR/DL": r"\bD\s*\([^)]*[RL]",
        "DC": r"\bD\s*\([^)]*C",
        "DM": r"\bDM\b",
        "MR/ML": r"\bM\s*\([^)]*[RL]",
        "MC": r"\bM\s*\([^)]*C",
        "AMR/AML": r"\bAM\s*\([^)]*[RL]",
        "AMC": r"\bAM\s*\([^)]*C",
        "STC": r"\bST\b",
    }
    return position_text.str.contains(patterns[position_key], regex=True, na=False)


def _role_attribute_score(players, role):
    profile = ROLE_PROFILES.get(role, {})
    available = {attribute: weight for attribute, weight in profile.items() if attribute in players.columns}
    if not available:
        return pd.Series(0.0, index=players.index)
    values = players[list(available)].apply(pd.to_numeric, errors="coerce").fillna(0).clip(0, 20)
    weights = pd.Series(available)
    return values.div(20).mul(weights, axis=1).sum(axis=1).div(weights.sum()).mul(100)


def recommend_players(budget, position, playstyle="balanced", top_n=10, min_confidence=0):
    position_key = POSITION_ALIASES.get(str(position).strip().upper())
    if position_key is None:
        raise ValueError(f"Posisi harus salah satu dari: {list(POSITION_ALIASES)}")
    playstyle_key = str(playstyle).strip().lower()
    if playstyle_key not in PLAYSTYLE_BONUSES:
        raise ValueError(f"Playstyle harus salah satu dari: {list(PLAYSTYLE_BONUSES)}")

    budget_values = pd.to_numeric(df["Transfer Value"], errors="coerce").fillna(0)
    candidates = df[(budget_values <= float(budget)) & _position_mask(df, position_key)].copy()
    if candidates.empty:
        return pd.DataFrame()

    role_scores = pd.DataFrame(index=candidates.index)
    for role in ROLE_BY_POSITION[position_key]:
        role_scores[role] = _role_attribute_score(candidates, role)
    candidates["Predicted Role"] = role_scores.idxmax(axis=1)
    candidates["Role Fit"] = role_scores.max(axis=1).round(2)
    candidates["Playstyle Fit"] = candidates["Predicted Role"].isin(PLAYSTYLE_BONUSES[playstyle_key]).astype(float).mul(100)

    model_features = candidates.reindex(columns=feature_columns, fill_value=0).apply(pd.to_numeric, errors="coerce").fillna(0)
    probabilities = model.predict_proba(model_features)
    candidates["Model Confidence"] = pd.Series(probabilities.max(axis=1) * 100, index=candidates.index).round(2)
    candidates["Recommendation Score"] = (
        0.70 * candidates["Role Fit"]
        + 0.20 * candidates["Playstyle Fit"]
        + 0.10 * candidates["Model Confidence"]
    ).round(2)

    result_columns = ["Name", "Club", "Transfer Value", "Position", "Predicted Role", "Role Fit", "Playstyle Fit", "Model Confidence", "Recommendation Score"]
    return candidates[candidates["Model Confidence"] >= min_confidence].sort_values("Recommendation Score", ascending=False)[result_columns].head(top_n)


recommendations = recommend_players(budget, position, playstyle, top_n)
print(recommendations.to_string(index=False))

                Name           Club  Transfer Value     Position       Predicted Role  Role Fit  Playstyle Fit  Model Confidence  Recommendation Score
     Frenkie de Jong      Barcelona      71000000.0    DM, M (C)              Regista     85.30            0.0         94.059998                 69.12
    Florentino LuÃ­s        Burnley      17000000.0    DM, M (C) Defensive Midfielder     80.33            0.0         91.809998                 65.41
AurÃ©lien Tchouameni    Real Madrid      53000000.0    DM, M (C) Defensive Midfielder     78.67            0.0         88.650002                 63.93
         Dani Parejo     Villarreal        650000.0 DM, M/AM (C)              Regista     76.82            0.0         96.440002                 63.42
   Stanislav Lobotka         Napoli      41000000.0    DM, M (C)            Half Back     77.26            0.0         93.250000                 63.41
   Rodrigo Bentancur      Tottenham      53000000.0    DM, M (C)            Half Back     77.4

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display


def search_players(position, role, budget_million, playstyle, top_n):
    results = recommend_players(
        budget=float(budget_million) * 1_000_000,
        position=position,
        playstyle=playstyle,
        top_n=top_n,
    )
    if role != "Semua role":
        results = results[results["Predicted Role"] == role]
    return results.reset_index(drop=True)


position_input = widgets.Dropdown(
    options=list(ROLE_BY_POSITION.keys()),
    value="STC",
    description="Posisi:",
    layout=widgets.Layout(width="260px"),
)
role_input = widgets.Dropdown(
    options=["Semua role"] + ROLE_BY_POSITION[position_input.value],
    value="Semua role",
    description="Role:",
    layout=widgets.Layout(width="320px"),
)
budget_input = widgets.FloatText(
    value=20,
    min=0,
    step=1,
    description="Budget (M):",
    layout=widgets.Layout(width="260px"),
)
playstyle_input = widgets.Dropdown(
    options=list(PLAYSTYLE_BONUSES.keys()),
    value="gegenpress",
    description="Gaya:",
    layout=widgets.Layout(width="260px"),
)
top_n_input = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description="Jumlah:",
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)
search_button = widgets.Button(
    description="Cari pemain",
    button_style="primary",
    icon="search",
)
output = widgets.Output()


def update_roles(change):
    role_input.options = ["Semua role"] + ROLE_BY_POSITION[change["new"]]
    role_input.value = "Semua role"


def run_search(_):
    with output:
        clear_output(wait=True)
        try:
            results = search_players(
                position_input.value,
                role_input.value,
                budget_input.value,
                playstyle_input.value,
                top_n_input.value,
            )
            if results.empty:
                print("Tidak ada pemain yang sesuai dengan filter.")
            else:
                display(results)
        except ValueError as error:
            print(f"Input tidak valid: {error}")


position_input.observe(update_roles, names="value")
search_button.on_click(run_search)

search_form = widgets.VBox([
    widgets.HBox([position_input, role_input]),
    widgets.HBox([budget_input, playstyle_input, top_n_input]),
    search_button,
    output,
])
display(search_form)
run_search(None)